In [1]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Add,
    GlobalAveragePooling1D,
    Concatenate,
    Lambda,
    Reshape
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
tf.keras.backend.clear_session()

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 2. target
# =====================================
y = df["label"].astype(int)

# =====================================
# 3. feature groups (LLaMA version)
# =====================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_score_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_score_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
    
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. 只跑 attention branch 不為空的組合
#    semantic + financial 進 attention
#    lexical 留在外面
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)

# =====================================
# 7. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# =====================================
# 8. 建立前處理器
#    輸出順序固定：
#    [semantic + financial] 在前（attention branch）
#    [lexical] 在後（aux branch）
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]

    transformers = []

    # attention branch: semantic
    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    # attention branch: financial
    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    # aux branch: lexical
    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    n_attn = len(sem_in_use) + len(fin_in_use)
    n_lex = len(lex_in_use)

    return preprocessor, n_attn, n_lex

# =====================================
# 9. Transformer block
# =====================================
def transformer_block(x, num_heads=2, ff_dim=32, dropout_rate=0.2):
    d_model = int(x.shape[-1])

    attn_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_model // num_heads)
    )(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    x = Add()([x, attn_output])
    x = LayerNormalization(epsilon=1e-6)(x)

    ff_output = Dense(ff_dim, activation="relu")(x)
    ff_output = Dropout(dropout_rate)(ff_output)
    ff_output = Dense(d_model)(ff_output)

    x = Add()([x, ff_output])
    x = LayerNormalization(epsilon=1e-6)(x)
    return x

# =====================================
# 10. semantic + financial attention model
#     full input 進來，在模型內部分 attn / lexical
# =====================================
def build_semfin_attention_model(
    meta,
    n_attn,
    units=16,
    d_model=16,
    num_heads=2,
    ff_dim=32,
    lex_dense=8,
    dropout_rate=0.2,
    learning_rate=0.001
):
    n_total_features = meta["n_features_in_"]
    n_lex = n_total_features - n_attn

    full_input = Input(shape=(n_total_features,), name="full_input")

    # attention slice: semantic + financial
    attn_part = Lambda(
        lambda x: x[:, :n_attn],
        output_shape=(n_attn,),
        name="attn_slice"
    )(full_input)
    attn_part = Reshape((n_attn, 1), name="attn_reshape")(attn_part)

    # attention branch: LSTM + Transformer
    x_attn = LSTM(units=units, return_sequences=True)(attn_part)
    x_attn = Dense(d_model)(x_attn)
    x_attn = transformer_block(
        x_attn,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout_rate=dropout_rate
    )
    x_attn = GlobalAveragePooling1D()(x_attn)

    # lexical branch
    if n_lex > 0:
        lex_part = Lambda(
            lambda x: x[:, n_attn:],
            output_shape=(n_lex,),
            name="lex_slice"
        )(full_input)
        x_lex = Dense(lex_dense, activation="relu")(lex_part)
        x_lex = Dropout(dropout_rate)(x_lex)
        x = Concatenate()([x_attn, x_lex])
    else:
        x = x_attn

    x = Dropout(dropout_rate)(x)
    x = Dense(16, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=full_input, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model

# =====================================
# 11. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]

# =====================================
# 12. 建立 pipeline
# =====================================
def build_semfin_attention_pipeline(selected_cols):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            n_attn=n_attn
        ))
    ])
    return pipeline

# =====================================
# 13. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__d_model": [16, 32],
    "classifier__model__num_heads": [2],
    "classifier__model__ff_dim": [32, 64],
    "classifier__model__lex_dense": [8, 16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}

# =====================================
# 14. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1

# =====================================
# 15. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    clean_params = {}
    for k, v in best_params.items():
        if k.startswith("classifier__model__"):
            clean_params[k.replace("classifier__model__", "")] = v

    batch_size = best_params["classifier__batch_size"]
    epochs = best_params["classifier__epochs"]

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            batch_size=batch_size,
            epochs=epochs,
            n_attn=n_attn,
            **clean_params
        ))
    ])
    return pipeline

# =====================================
# 16. outer 10-fold evaluation
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    grid_search = GridSearchCV(
        estimator=build_semfin_attention_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True,
        error_score="raise"
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        tf.keras.backend.clear_session()

        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        model = build_best_pipeline(X.columns.tolist(), best_params)
        model.fit(X_train_sub, y_train_sub)

        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, _ = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    return {
        "Model": "SemFinAttention_LSTM_Transformer",
        "Feature_Set": feature_set_name,
        "Num_Features": X.shape[1],
        "Accuracy": np.mean(metrics_df["accuracy"]),
        "F1": np.mean(metrics_df["f1"]),
        "ROC_AUC": np.mean(metrics_df["roc_auc"]),
        "Precision": np.mean(metrics_df["precision"]),
        "Recall": np.mean(metrics_df["recall"]),
        "PR_AUC": np.mean(metrics_df["average_precision"]),
        "Mean_Best_Threshold": np.mean(best_thresholds)
    }

# =====================================
# 17. 執行全部
# =====================================
all_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    result = evaluate_feature_set(X, y, feature_set_name)
    all_results.append(result)

# =====================================
# 18. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\nFinal Results:")
print(results_df)

# =====================================
# 19. 輸出
# =====================================
output_file = "llama_semantic_financial_attention_LSTM_Transformer_M1_M3_M4_M5_M6_nestedCV_threshold_tuned.csv"
results_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_file}")

Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__d_model': 32, 'classifier__model__dropout_rate': 0.2, 'classifier__model__ff_dim': 64, 'classifier__model__learning_rate': 0.001, 'classifier__model__lex_dense': 8, 'classifier__model__num_heads': 2, 'classifier__model__units': 16}
  Fold 01 | threshold=0.85 | F1=0.7500 | ROC_AUC=0.8448
  Fold 02 | threshold=0.45 | F1=0.7273 | ROC_AUC=0.9741
  Fold 03 | threshold=0.80 | F1=0.0000 | ROC_AUC=0.9000
  Fold 04 | threshold=0.80 | F1=0.7500 | ROC_AUC=1.0000
  Fold 05 | threshold=0.85 | F1=0.6667 | ROC_AUC=0.9889
  Fold 06 | threshold=0.75 | F1=0.3333 | ROC_AUC=0.7667
  Fold 07 | threshold=0.70 | F1=0.5000 | ROC_AUC=0.8222
  Fold 08 | threshold=0.90 | F1=0.8000 | ROC_AUC=1.0000
  Fold 09 | threshold=0.90 | F1=0.2857 | ROC_AUC=0.9080
  Fold 10 | threshold=0.45 | F1=0.5714 | ROC_AUC

In [2]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Add,
    GlobalAveragePooling1D,
    Concatenate,
    Lambda,
    Reshape
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
tf.keras.backend.clear_session()

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 2. target
# =====================================
y = df["label"].astype(int)

# =====================================
# 3. feature groups (LLaMA version)
# =====================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_score_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_score_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
    
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. 只跑 attention branch 不為空的組合
#    semantic + financial 進 attention
#    lexical 留在外面
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)

# =====================================
# 7. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# =====================================
# 8. 建立前處理器
#    輸出順序固定：
#    [semantic + financial] 在前（attention branch）
#    [lexical] 在後（aux branch）
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]

    transformers = []

    # attention branch: semantic
    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    # attention branch: financial
    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    # aux branch: lexical
    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    n_attn = len(sem_in_use) + len(fin_in_use)
    n_lex = len(lex_in_use)

    return preprocessor, n_attn, n_lex

# =====================================
# 9. Transformer block
# =====================================
def transformer_block(x, num_heads=2, ff_dim=32, dropout_rate=0.2):
    d_model = int(x.shape[-1])

    attn_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_model // num_heads)
    )(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    x = Add()([x, attn_output])
    x = LayerNormalization(epsilon=1e-6)(x)

    ff_output = Dense(ff_dim, activation="relu")(x)
    ff_output = Dropout(dropout_rate)(ff_output)
    ff_output = Dense(d_model)(ff_output)

    x = Add()([x, ff_output])
    x = LayerNormalization(epsilon=1e-6)(x)
    return x

# =====================================
# 10. semantic + financial attention model
#     full input 進來，在模型內部分 attn / lexical
# =====================================
def build_semfin_attention_model(
    meta,
    n_attn,
    units=16,
    d_model=16,
    num_heads=2,
    ff_dim=32,
    lex_dense=8,
    dropout_rate=0.2,
    learning_rate=0.001
):
    n_total_features = meta["n_features_in_"]
    n_lex = n_total_features - n_attn

    full_input = Input(shape=(n_total_features,), name="full_input")

    # attention slice: semantic + financial
    attn_part = Lambda(
        lambda x: x[:, :n_attn],
        output_shape=(n_attn,),
        name="attn_slice"
    )(full_input)
    attn_part = Reshape((n_attn, 1), name="attn_reshape")(attn_part)

    # attention branch: LSTM + Transformer
    x_attn = LSTM(units=units, return_sequences=True)(attn_part)
    x_attn = Dense(d_model)(x_attn)
    x_attn = transformer_block(
        x_attn,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout_rate=dropout_rate
    )
    x_attn = GlobalAveragePooling1D()(x_attn)

    # lexical branch
    if n_lex > 0:
        lex_part = Lambda(
            lambda x: x[:, n_attn:],
            output_shape=(n_lex,),
            name="lex_slice"
        )(full_input)
        x_lex = Dense(lex_dense, activation="relu")(lex_part)
        x_lex = Dropout(dropout_rate)(x_lex)
        x = Concatenate()([x_attn, x_lex])
    else:
        x = x_attn

    x = Dropout(dropout_rate)(x)
    x = Dense(16, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=full_input, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model

# =====================================
# 11. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]

# =====================================
# 12. 建立 pipeline
# =====================================
def build_semfin_attention_pipeline(selected_cols):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            n_attn=n_attn
        ))
    ])
    return pipeline

# =====================================
# 13. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__d_model": [16, 32],
    "classifier__model__num_heads": [2],
    "classifier__model__ff_dim": [32, 64],
    "classifier__model__lex_dense": [8, 16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}

# =====================================
# 14. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1

# =====================================
# 15. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    clean_params = {}
    for k, v in best_params.items():
        if k.startswith("classifier__model__"):
            clean_params[k.replace("classifier__model__", "")] = v

    batch_size = best_params["classifier__batch_size"]
    epochs = best_params["classifier__epochs"]

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            batch_size=batch_size,
            epochs=epochs,
            n_attn=n_attn,
            **clean_params
        ))
    ])
    return pipeline

# =====================================
# 16. outer 10-fold evaluation
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    grid_search = GridSearchCV(
        estimator=build_semfin_attention_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True,
        error_score="raise"
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        tf.keras.backend.clear_session()

        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        model = build_best_pipeline(X.columns.tolist(), best_params)
        model.fit(X_train_sub, y_train_sub)

        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, _ = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    return {
        "Model": "SemFinAttention_LSTM_Transformer",
        "Feature_Set": feature_set_name,
        "Num_Features": X.shape[1],
        "Accuracy": np.mean(metrics_df["accuracy"]),
        "F1": np.mean(metrics_df["f1"]),
        "ROC_AUC": np.mean(metrics_df["roc_auc"]),
        "Precision": np.mean(metrics_df["precision"]),
        "Recall": np.mean(metrics_df["recall"]),
        "PR_AUC": np.mean(metrics_df["average_precision"]),
        "Mean_Best_Threshold": np.mean(best_thresholds)
    }

# =====================================
# 17. 執行全部
# =====================================
all_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    result = evaluate_feature_set(X, y, feature_set_name)
    all_results.append(result)

# =====================================
# 18. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\nFinal Results:")
print(results_df)

# =====================================
# 19. 輸出
# =====================================
output_file = "chatgpt_semantic_financial_attention_LSTM_Transformer_M1_M3_M4_M5_M6_nestedCV_threshold_tuned.csv"
results_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_file}")

Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__d_model': 16, 'classifier__model__dropout_rate': 0.2, 'classifier__model__ff_dim': 64, 'classifier__model__learning_rate': 0.001, 'classifier__model__lex_dense': 8, 'classifier__model__num_heads': 2, 'classifier__model__units': 16}
  Fold 01 | threshold=0.90 | F1=0.6667 | ROC_AUC=0.8103
  Fold 02 | threshold=0.30 | F1=1.0000 | ROC_AUC=1.0000
  Fold 03 | threshold=0.60 | F1=0.8571 | ROC_AUC=0.9889
  Fold 04 | threshold=0.90 | F1=0.6667 | ROC_AUC=0.9778
  Fold 05 | threshold=0.50 | F1=0.8571 | ROC_AUC=1.0000
  Fold 06 | threshold=0.90 | F1=0.6667 | ROC_AUC=0.9000
  Fold 07 | threshold=0.50 | F1=0.5000 | ROC_AUC=0.9000
  Fold 08 | threshold=0.75 | F1=0.6667 | ROC_AUC=0.9333
  Fold 09 | threshold=0.50 | F1=0.5455 | ROC_AUC=0.9080
  Fold 10 | threshold=0.40 | F1=0.5455 | ROC_AUC

In [1]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Add,
    GlobalAveragePooling1D,
    Concatenate,
    Lambda,
    Reshape
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
tf.keras.backend.clear_session()

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 2. target
# =====================================
y = df["label"].astype(int)

# =====================================
# 3. 反轉反向指標（LLaMA version）
# 分數越高 -> 越漂綠 / 風險越高
# =====================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

print("✅ Llama 反向指標已建立")

# =====================================
# 4. feature groups (LLaMA version)
# 注意：這裡改用 risk 欄位
# =====================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 5. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 6. 只跑 attention branch 不為空的組合
#    semantic + financial 進 attention
#    lexical 留在外面
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 7. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)

# =====================================
# 8. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# =====================================
# 9. 建立前處理器
#    輸出順序固定：
#    [semantic + financial] 在前（attention branch）
#    [lexical] 在後（aux branch）
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]

    transformers = []

    # attention branch: semantic
    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    # attention branch: financial
    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    # aux branch: lexical
    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    n_attn = len(sem_in_use) + len(fin_in_use)
    n_lex = len(lex_in_use)

    return preprocessor, n_attn, n_lex

# =====================================
# 10. Transformer block
# =====================================
def transformer_block(x, num_heads=2, ff_dim=32, dropout_rate=0.2):
    d_model = int(x.shape[-1])

    attn_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_model // num_heads)
    )(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    x = Add()([x, attn_output])
    x = LayerNormalization(epsilon=1e-6)(x)

    ff_output = Dense(ff_dim, activation="relu")(x)
    ff_output = Dropout(dropout_rate)(ff_output)
    ff_output = Dense(d_model)(ff_output)

    x = Add()([x, ff_output])
    x = LayerNormalization(epsilon=1e-6)(x)
    return x

# =====================================
# 11. semantic + financial attention model
#     full input 進來，在模型內部分 attn / lexical
# =====================================
def build_semfin_attention_model(
    meta,
    n_attn,
    units=16,
    d_model=16,
    num_heads=2,
    ff_dim=32,
    lex_dense=8,
    dropout_rate=0.2,
    learning_rate=0.001
):
    n_total_features = meta["n_features_in_"]
    n_lex = n_total_features - n_attn

    full_input = Input(shape=(n_total_features,), name="full_input")

    # attention slice: semantic + financial
    attn_part = Lambda(
        lambda x: x[:, :n_attn],
        output_shape=(n_attn,),
        name="attn_slice"
    )(full_input)
    attn_part = Reshape((n_attn, 1), name="attn_reshape")(attn_part)

    # attention branch: LSTM + Transformer
    x_attn = LSTM(units=units, return_sequences=True)(attn_part)
    x_attn = Dense(d_model)(x_attn)
    x_attn = transformer_block(
        x_attn,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout_rate=dropout_rate
    )
    x_attn = GlobalAveragePooling1D()(x_attn)

    # lexical branch
    if n_lex > 0:
        lex_part = Lambda(
            lambda x: x[:, n_attn:],
            output_shape=(n_lex,),
            name="lex_slice"
        )(full_input)
        x_lex = Dense(lex_dense, activation="relu")(lex_part)
        x_lex = Dropout(dropout_rate)(x_lex)
        x = Concatenate()([x_attn, x_lex])
    else:
        x = x_attn

    x = Dropout(dropout_rate)(x)
    x = Dense(16, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=full_input, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model

# =====================================
# 12. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]

# =====================================
# 13. 建立 pipeline
# =====================================
def build_semfin_attention_pipeline(selected_cols):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            n_attn=n_attn
        ))
    ])
    return pipeline

# =====================================
# 14. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__d_model": [16, 32],
    "classifier__model__num_heads": [2],
    "classifier__model__ff_dim": [32, 64],
    "classifier__model__lex_dense": [8, 16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}

# =====================================
# 15. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1

# =====================================
# 16. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    clean_params = {}
    for k, v in best_params.items():
        if k.startswith("classifier__model__"):
            clean_params[k.replace("classifier__model__", "")] = v

    batch_size = best_params["classifier__batch_size"]
    epochs = best_params["classifier__epochs"]

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            batch_size=batch_size,
            epochs=epochs,
            n_attn=n_attn,
            **clean_params
        ))
    ])
    return pipeline

# =====================================
# 17. outer 10-fold evaluation
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    grid_search = GridSearchCV(
        estimator=build_semfin_attention_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True,
        error_score="raise"
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        tf.keras.backend.clear_session()

        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        model = build_best_pipeline(X.columns.tolist(), best_params)
        model.fit(X_train_sub, y_train_sub)

        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, _ = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    return {
        "Model": "SemFinAttention_LSTM_Transformer",
        "Feature_Set": feature_set_name,
        "Config": str(best_params),

        "Num_Features": X.shape[1],
        "Num_Continuous": len([c for c in X.columns if c in semantic_cols + financial_cols]),
        "Num_Lexical": len([c for c in X.columns if c in lexical_cols]),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Threshold_std": np.std(best_thresholds)
    }

# =====================================
# 18. 執行全部
# =====================================
all_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    result = evaluate_feature_set(X, y, feature_set_name)
    all_results.append(result)

# =====================================
# 19. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\nFinal Results:")
print(results_df)

# =====================================
# 20. 輸出
# =====================================
output_file = "llama_semantic_financial_attention_LSTM_Transformer_M1_M3_M4_M5_M6_nestedCV_threshold_tuned_with_std.csv"
results_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_file}")

✅ Llama 反向指標已建立
Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__d_model': 16, 'classifier__model__dropout_rate': 0.2, 'classifier__model__ff_dim': 64, 'classifier__model__learning_rate': 0.001, 'classifier__model__lex_dense': 8, 'classifier__model__num_heads': 2, 'classifier__model__units': 16}
  Fold 01 | threshold=0.85 | F1=0.6667 | ROC_AUC=0.9052
  Fold 02 | threshold=0.60 | F1=0.7500 | ROC_AUC=0.9741
  Fold 03 | threshold=0.90 | F1=0.0000 | ROC_AUC=0.9222
  Fold 04 | threshold=0.65 | F1=0.4444 | ROC_AUC=0.9444
  Fold 05 | threshold=0.90 | F1=0.6667 | ROC_AUC=0.9889
  Fold 06 | threshold=0.85 | F1=0.0000 | ROC_AUC=0.7889
  Fold 07 | threshold=0.55 | F1=0.6000 | ROC_AUC=0.9556
  Fold 08 | threshold=0.85 | F1=1.0000 | ROC_AUC=1.0000
  Fold 09 | threshold=0.45 | F1=0.6667 | ROC_AUC=0.9310
  Fold 10 | threshold=0.85 | F1=

In [2]:
#chatgpt
import os
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Add,
    GlobalAveragePooling1D,
    Concatenate,
    Lambda,
    Reshape
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
tf.keras.backend.clear_session()

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 2. target
# =====================================
y = df["label"].astype(int)

# =====================================
# 3. 反轉反向指標（LLaMA version）
# 分數越高 -> 越漂綠 / 風險越高
# =====================================
reverse_map = {
    "chatgpt_vagueness_score_1": "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1": "chatgpt_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

print("✅ ChatGPT 反向指標已建立")

# =====================================
# 4. feature groups (ChatGPT version)
# 注意：這裡改用 risk 欄位
# =====================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_risk_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_risk_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 5. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 6. 只跑 attention branch 不為空的組合
#    semantic + financial 進 attention
#    lexical 留在外面
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 7. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)

# =====================================
# 8. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# =====================================
# 9. 建立前處理器
#    輸出順序固定：
#    [semantic + financial] 在前（attention branch）
#    [lexical] 在後（aux branch）
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]

    transformers = []

    # attention branch: semantic
    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    # attention branch: financial
    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    # aux branch: lexical
    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    n_attn = len(sem_in_use) + len(fin_in_use)
    n_lex = len(lex_in_use)

    return preprocessor, n_attn, n_lex

# =====================================
# 10. Transformer block
# =====================================
def transformer_block(x, num_heads=2, ff_dim=32, dropout_rate=0.2):
    d_model = int(x.shape[-1])

    attn_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_model // num_heads)
    )(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    x = Add()([x, attn_output])
    x = LayerNormalization(epsilon=1e-6)(x)

    ff_output = Dense(ff_dim, activation="relu")(x)
    ff_output = Dropout(dropout_rate)(ff_output)
    ff_output = Dense(d_model)(ff_output)

    x = Add()([x, ff_output])
    x = LayerNormalization(epsilon=1e-6)(x)
    return x

# =====================================
# 11. semantic + financial attention model
#     full input 進來，在模型內部分 attn / lexical
# =====================================
def build_semfin_attention_model(
    meta,
    n_attn,
    units=16,
    d_model=16,
    num_heads=2,
    ff_dim=32,
    lex_dense=8,
    dropout_rate=0.2,
    learning_rate=0.001
):
    n_total_features = meta["n_features_in_"]
    n_lex = n_total_features - n_attn

    full_input = Input(shape=(n_total_features,), name="full_input")

    # attention slice: semantic + financial
    attn_part = Lambda(
        lambda x: x[:, :n_attn],
        output_shape=(n_attn,),
        name="attn_slice"
    )(full_input)
    attn_part = Reshape((n_attn, 1), name="attn_reshape")(attn_part)

    # attention branch: LSTM + Transformer
    x_attn = LSTM(units=units, return_sequences=True)(attn_part)
    x_attn = Dense(d_model)(x_attn)
    x_attn = transformer_block(
        x_attn,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout_rate=dropout_rate
    )
    x_attn = GlobalAveragePooling1D()(x_attn)

    # lexical branch
    if n_lex > 0:
        lex_part = Lambda(
            lambda x: x[:, n_attn:],
            output_shape=(n_lex,),
            name="lex_slice"
        )(full_input)
        x_lex = Dense(lex_dense, activation="relu")(lex_part)
        x_lex = Dropout(dropout_rate)(x_lex)
        x = Concatenate()([x_attn, x_lex])
    else:
        x = x_attn

    x = Dropout(dropout_rate)(x)
    x = Dense(16, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=full_input, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model

# =====================================
# 12. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]

# =====================================
# 13. 建立 pipeline
# =====================================
def build_semfin_attention_pipeline(selected_cols):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            n_attn=n_attn
        ))
    ])
    return pipeline

# =====================================
# 14. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__d_model": [16, 32],
    "classifier__model__num_heads": [2],
    "classifier__model__ff_dim": [32, 64],
    "classifier__model__lex_dense": [8, 16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}

# =====================================
# 15. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1

# =====================================
# 16. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    preprocessor, n_attn, n_lex = build_preprocessor(selected_cols)

    clean_params = {}
    for k, v in best_params.items():
        if k.startswith("classifier__model__"):
            clean_params[k.replace("classifier__model__", "")] = v

    batch_size = best_params["classifier__batch_size"]
    epochs = best_params["classifier__epochs"]

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", KerasClassifier(
            model=build_semfin_attention_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            batch_size=batch_size,
            epochs=epochs,
            n_attn=n_attn,
            **clean_params
        ))
    ])
    return pipeline

# =====================================
# 17. outer 10-fold evaluation
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    grid_search = GridSearchCV(
        estimator=build_semfin_attention_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True,
        error_score="raise"
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        tf.keras.backend.clear_session()

        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        model = build_best_pipeline(X.columns.tolist(), best_params)
        model.fit(X_train_sub, y_train_sub)

        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, _ = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    return {
        "Model": "SemFinAttention_LSTM_Transformer",
        "Feature_Set": feature_set_name,
        "Config": str(best_params),

        "Num_Features": X.shape[1],
        "Num_Continuous": len([c for c in X.columns if c in semantic_cols + financial_cols]),
        "Num_Lexical": len([c for c in X.columns if c in lexical_cols]),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Threshold_std": np.std(best_thresholds)
    }

# =====================================
# 18. 執行全部
# =====================================
all_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    result = evaluate_feature_set(X, y, feature_set_name)
    all_results.append(result)

# =====================================
# 19. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\nFinal Results:")
print(results_df)

# =====================================
# 20. 輸出
# =====================================
output_file = "chatgpt_semantic_financial_attention_LSTM_Transformer_M1_M3_M4_M5_M6_nestedCV_threshold_tuned_with_std.csv"
results_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_file}")

✅ ChatGPT 反向指標已建立
Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__d_model': 16, 'classifier__model__dropout_rate': 0.2, 'classifier__model__ff_dim': 64, 'classifier__model__learning_rate': 0.001, 'classifier__model__lex_dense': 8, 'classifier__model__num_heads': 2, 'classifier__model__units': 16}
  Fold 01 | threshold=0.90 | F1=0.7500 | ROC_AUC=0.9397
  Fold 02 | threshold=0.20 | F1=0.8889 | ROC_AUC=1.0000
  Fold 03 | threshold=0.20 | F1=0.7500 | ROC_AUC=0.9889
  Fold 04 | threshold=0.80 | F1=0.8571 | ROC_AUC=0.9778
  Fold 05 | threshold=0.80 | F1=0.8571 | ROC_AUC=0.9889
  Fold 06 | threshold=0.85 | F1=0.0000 | ROC_AUC=0.8667
  Fold 07 | threshold=0.90 | F1=0.6000 | ROC_AUC=0.9556
  Fold 08 | threshold=0.85 | F1=0.8571 | ROC_AUC=1.0000
  Fold 09 | threshold=0.85 | F1=0.4000 | ROC_AUC=0.8736
  Fold 10 | threshold=0.70 | F